💡 **Environment:** `clamp-analyses`

# Supplementary Figure 2: Donor-bulk recovery and cell-type gene-module maps

In [ ]:
suppressPackageStartupMessages({
  library(here)
})
source(here("scripts/panels/style.R"))
source(here::here("scripts/panels/gm_ids.R"))


In [ ]:
# Original analysis and provenance guards retained; layout/export lives in supp2.R.
panel_input <- list(
  purity = here::here("output/03_model_biology/00_pseudobulk/06_donor_bulk_recovery/supp2_purity.csv"),
  umap_cells = here::here("output/03_model_biology/00_pseudobulk/06_donor_bulk_recovery/supp2_umap_cells.csv.gz"),
  umap_lvs = here::here("output/03_model_biology/00_pseudobulk/06_donor_bulk_recovery/supp2_umap_lvs.csv"),
  donor_bulk_recovery = here::here("output/03_model_biology/00_pseudobulk/06_donor_bulk_recovery/analysis/single_cell_recovery.csv"),
  donor_bulk_overall = here::here("output/03_model_biology/00_pseudobulk/06_donor_bulk_recovery/analysis/single_cell_recovery_overall.csv")
)
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
  library(cowplot)
  library(ggrastr)
  library(ggrepel)
  library(grid)
  library(svglite)
  library(ragg)
})


FIG_W <- 183
FIG_H <- 445
FONT_FAMILY <- "Helvetica"
FS_TAG <- 8
FS_TITLE <- 6.5
FS_AXIS_TITLE <- 6.5
FS_AXIS_TEXT <- 5
LINE_W <- 0.25

DATASET_LEVELS <- c("Heart_Datar2026", "PBMC_1k1k",
  "Brain_Mathys2023", "Brain_Xiong2023",
  "Lung_Sikkema2023", "PBMC_Perez2022")
DATASET_LABELS <- c(Heart_Datar2026="Heart: Datar", PBMC_1k1k="PBMC: 1k1k",
  Brain_Mathys2023="Brain: Mathys", Brain_Xiong2023="Brain: Xiong",
  Lung_Sikkema2023="Lung: Sikkema", PBMC_Perez2022="PBMC: Perez")
DATASET_COLORS <- c(Brain_Mathys2023="#5B8FF9", Brain_Xiong2023="#9270CA",
  Heart_Datar2026="#E8684A", PBMC_1k1k="#5AD8A6",
  PBMC_Perez2022="#F6BD16", Lung_Sikkema2023="#6DC8EC")
KNOWN_CELL_COLORS <- c(B_cell="#4477AA", Myeloid="#EE6677",
  NK="#228833", T_cell="#CCBB44", CD4_T="#66CCEE",
  CD8_T="#AA3377", CD14_Mono="#EE7733", CD16_Mono="#0077BB",
  DC="#BBBBBB", Plasma_B="#EE3377", gd_T="#009988")

purity <- fread_gm(panel_input[["purity"]])
cells <- fread_gm(panel_input[["umap_cells"]])
lvs <- fread_gm(panel_input[["umap_lvs"]])
donor_bulk_recovery <- fread_gm(panel_input[["donor_bulk_recovery"]])
donor_bulk_overall <- fread_gm(panel_input[["donor_bulk_overall"]])
stopifnot(grepl("/06_donor_bulk_recovery/", panel_input[["purity"]], fixed=TRUE),
          grepl("/06_donor_bulk_recovery/", panel_input[["umap_lvs"]], fixed=TRUE),
          grepl("/06_donor_bulk_recovery/", panel_input[["donor_bulk_recovery"]], fixed=TRUE),
          grepl("/06_donor_bulk_recovery/", panel_input[["donor_bulk_overall"]], fixed=TRUE),
          nrow(donor_bulk_recovery) == 47L)

# Hard provenance guard for Supplementary Fig. 2a. These are donor-bulk
# projection results, distinct from the pseudobulk values in Supp. Fig. 1e.
assert_close <- function(actual, expected, tolerance, label) {
  if (length(actual) != 1L || !is.finite(actual) || abs(actual - expected) > tolerance)
    stop(sprintf("%s provenance check failed: observed %.6f, expected %.6f +/- %.6f",
                 label, actual, expected, tolerance))
}
assert_close(donor_bulk_overall$global_recovery_pct_pooled, 70.618975, 0.05, "donor-bulk pooled purity (%)")
assert_close(donor_bulk_overall$mean_recovery_pct_by_dataset, 77.455512, 0.05, "donor-bulk mean dataset purity (%)")
assert_close(donor_bulk_overall$mean_purity_ratio, 0.781929, 0.001, "donor-bulk mean purity ratio")
assert_close(donor_bulk_overall$mean_purity_lift, 19.595991, 0.05, "donor-bulk mean purity lift")
assert_close(sum(donor_bulk_recovery$diag_count) / sum(donor_bulk_recovery$n_top) * 100,
             donor_bulk_overall$global_recovery_pct_pooled, 1e-6, "donor-bulk pooled purity recomputation")
perez_b <- donor_bulk_recovery[dataset == "PBMC_Perez2022" & cell_type == "B_cell"]
stopifnot(nrow(perez_b) == 1L, perez_b$assigned_lv == "GM32",
          setequal(paste(purity$dataset, purity$cell_type),
                   paste(donor_bulk_recovery$dataset, donor_bulk_recovery$cell_type)))
activity_columns <- grep("^activity_[0-9]+$", names(cells), value=TRUE)
activity_columns <- activity_columns[order(as.integer(sub("activity_", "", activity_columns)))]
stopifnot(setequal(unique(cells$dataset), DATASET_LEVELS),
          setequal(unique(lvs$dataset), DATASET_LEVELS),
          length(activity_columns) == max(lvs$plot_order),
          lvs[order(plot_order), all(plot_order == seq_len(.N)), by=dataset]$V1,
          cells[, all(uniqueN(cell_index) == .N), by=dataset]$V1,
          all(is.finite(as.matrix(cells[, .(umap1, umap2)]))))
for (dataset_id in DATASET_LEVELS) {
  selected_columns <- paste0("activity_", seq_len(lvs[dataset == dataset_id, .N]))
  stopifnot(all(is.finite(as.matrix(cells[dataset == dataset_id,
    selected_columns, with=FALSE]))))
}

all_cell_types <- sort(unique(cells$mapped_cell_type))
CELL_COLORS <- setNames(grDevices::hcl.colors(length(all_cell_types), "Dark 3"),
                        all_cell_types)
known <- intersect(names(KNOWN_CELL_COLORS), names(CELL_COLORS))
CELL_COLORS[known] <- KNOWN_CELL_COLORS[known]
pretty_cell_type <- function(x) sub("^Lymphatic Endothelial$", "Lymphatic EC",
  gsub("([a-z])([A-Z])", "\\1 \\2", gsub("_", " ", x, fixed=TRUE)))

theme_nm <- function() {
  theme_classic(base_size=FS_AXIS_TEXT, base_family=FONT_FAMILY) %+replace%
    theme(axis.line=element_line(linewidth=LINE_W, colour="black"),
      axis.ticks=element_line(linewidth=LINE_W, colour="black"),
      axis.text=element_text(size=FS_AXIS_TEXT, colour="black"),
      axis.title=element_text(size=FS_AXIS_TITLE, colour="black"),
      panel.grid=element_blank(), plot.margin=margin(1,1,1,1,"mm"))
}
add_tag <- function(p, label) {
  ggdraw() + draw_plot(p) +
    draw_label(label, x=0, y=1, hjust=0, vjust=1, size=FS_TAG,
               fontface="bold", fontfamily=FONT_FAMILY)
}

# A: top-1% annotated-cell purity across all matched LVs.
purity[, dataset := factor(dataset, levels=DATASET_LEVELS)]
purity[, dataset_label := factor(DATASET_LABELS[as.character(dataset)],
                                 levels=DATASET_LABELS[DATASET_LEVELS])]
purity_means <- purity[, .(mean_pct=mean(recovery_pct), max_pct=max(recovery_pct)),
                       by=.(dataset, dataset_label)]
panel_A <- ggplot(purity, aes(dataset_label, recovery_pct, fill=dataset)) +
  geom_violin(width=0.9, trim=TRUE, scale="width", colour="black", linewidth=0.22, alpha=0.65) +
  geom_boxplot(width=0.18, outlier.shape=NA, colour="black", linewidth=0.22, alpha=0.85) +
  geom_jitter(width=0.09, size=0.4, shape=21, fill="white",
              colour="#333333", stroke=0.12, alpha=0.75) +
  geom_point(data=purity_means, aes(x=dataset_label, y=mean_pct), shape=23,
             size=1.05, fill="white", colour="black", stroke=0.28, inherit.aes=FALSE) +
  geom_text(data=purity_means, aes(x=dataset_label, y=max_pct + 4,
                                   label=sprintf("%.1f", mean_pct)),
            size=MIN_FONT_PT/.pt, fontface="bold", family=FONT_FAMILY,
            vjust=0, inherit.aes=FALSE) +
  scale_fill_manual(values=DATASET_COLORS, drop=FALSE) +
  scale_y_continuous(breaks=seq(0,100,25), expand=expansion(mult=c(0.01,0.01))) +
  coord_cartesian(ylim=c(0, 112), clip="off") +
  labs(x=NULL, y="Top-1% annotated-cell purity (%)") +
  theme_nm() +
  theme(legend.position="none",
        axis.text.x=element_text(angle=25, hjust=1, size=5.5))

map_theme <- theme_void(base_family=FONT_FAMILY, base_size=4.2) +
  theme(panel.border=element_rect(colour="black", fill=NA, linewidth=0.25),
        plot.title=element_text(size=4.0, hjust=0.5, lineheight=0.92,
                                margin=margin(0,0,0.35,0,"mm")),
        plot.margin=margin(0.35,0.35,0.35,0.35,"mm"))

square_limits <- function(d, pad=0.04) {
  xr <- range(d$umap1, finite=TRUE)
  yr <- range(d$umap2, finite=TRUE)
  span <- max(diff(xr), diff(yr)) * (1 + 2 * pad)
  list(x=mean(xr) + c(-0.5, 0.5) * span,
       y=mean(yr) + c(-0.5, 0.5) * span)
}

feature_sheet <- function(dataset_id) {
  d <- cells[dataset == dataset_id]
  meta <- lvs[dataset == dataset_id][order(plot_order)]
  stopifnot(nrow(meta) > 0L, nrow(d) > 0L)
  limits <- square_limits(d)
  centroids <- d[, .(umap1=median(umap1), umap2=median(umap2)),
                 by=mapped_cell_type]
  centroids[, label := pretty_cell_type(mapped_cell_type)]
  annotation <- ggplot(d, aes(umap1, umap2, colour=mapped_cell_type)) +
    ggrastr::rasterise(geom_point(size=0.16, alpha=0.78), dpi=300) +
    ggrepel::geom_text_repel(data=centroids, aes(label=label),
      colour="black", size=1.20, fontface="bold", family=FONT_FAMILY,
      seed=123, box.padding=0.10, point.padding=0.05, min.segment.length=0,
      segment.size=0.16, max.overlaps=Inf, show.legend=FALSE) +
    scale_colour_manual(values=CELL_COLORS) +
    coord_equal(xlim=limits$x, ylim=limits$y, expand=FALSE) +
    labs(title="Cell-type annotation") + map_theme +
    theme(legend.position="none",
          plot.title=element_text(size=4.5, face="bold"))
  features <- lapply(seq_len(nrow(meta)), function(i) {
    m <- meta[plot_order == i]
    high <- CELL_COLORS[[m$cell_type]]
    score_col <- paste0("activity_", i)
    ggplot(d, aes(umap1, umap2, colour=.data[[score_col]])) +
      ggrastr::rasterise(geom_point(size=0.16), dpi=300) +
      scale_colour_gradientn(colours=c("#D8E2EF", "white", high)) +
      coord_equal(xlim=limits$x, ylim=limits$y, expand=FALSE) +
      labs(title=sprintf("%s · %s", pretty_cell_type(m$cell_type), m$LV)) +
      map_theme + theme(legend.position="none")
  })
  feature_grid <- plot_grid(plotlist=features, ncol=5, align="hv")
  map_row <- plot_grid(annotation, feature_grid, nrow=1,
                       rel_widths=c(0.30, 0.70))
  header <- ggdraw() + draw_label(DATASET_LABELS[[dataset_id]],
    x=0.5, y=0.45, hjust=0.5, size=FS_TITLE, fontface="bold",
    fontfamily=FONT_FAMILY)
  plot_grid(header, map_row, ncol=1, rel_heights=c(0.08, 0.92))
}

In [ ]:
set.seed(123)
nm_sources(2,unlist(panel_input))
UMAP_FONT <- MIN_FONT_PT
map_axes_theme <- theme_classic(base_family="Helvetica", base_size=UMAP_FONT) +
  theme(legend.position="none", axis.ticks.length=unit(.45,"mm"),
        axis.line=element_line(linewidth=.25), axis.ticks=element_line(linewidth=.25),
        axis.text=element_text(size=UMAP_FONT, colour="black"),
        axis.title=element_text(size=UMAP_FONT, colour="black"),
        plot.title=element_text(size=UMAP_FONT, face="bold", hjust=.5, lineheight=.95,
                                margin=margin(0,0,.4,0,"mm")),
        plot.margin=margin(.5,.5,.5,.5,"mm"))
hex_mean_layers <- function(x, y, activity, lim, col) {
  h <- hexbin::hexbin(x, y, xbins=150, xbnds=lim$x, ybnds=lim$y, IDs=TRUE)
  agg <- data.table(cell=h@cID, activity=activity)[, .(m=mean(activity), n=.N), by=cell]
  stopifnot(sum(agg$n) == length(x))
  xy <- hexbin::hcell2xy(h); hx <- data.table(cell=h@cell, umap1=xy$x, umap2=xy$y)[agg, on="cell"]
  list(ggrastr::rasterise(geom_hex(data=hx, aes(umap1, umap2, fill=m, colour=m), stat="identity", linewidth=.08), dpi=400),
       scale_fill_gradientn(colours=c("#E3E8EF", col), limits=quantile(hx[n >= 5L]$m, c(.5, .99)),
                            oob=scales::squish, guide="none", aesthetics=c("fill", "colour")))
}
DISTINCT_COLORS <- c(
  Adipocyte="#C9A400", `Atrial CM`="#F58231", `Ventricular CM`="#E6194B", Endocardial="#911EB4",
  Endothelial="#3CB44B", `Lymphatic EC`="#42D4F4", Epicardial="#808000", Fibroblast="#9A6324",
  Pericyte="#F032E6", VSMC="#800000", Lymphocyte="#4363D8", Macrophage="#469990", Mast="#000075",
  Neuronal="#222222",
  `Airway epithelium`="#E6194B", `Alveolar epithelium`="#F58231", `Blood vessels`="#911EB4",
  `Fibroblast lineage`="#9A6324", `Hematopoietic stem cells`="#C9A400",
  Lymphoid="#4363D8", Mesothelium="#808000", Myeloid="#3CB44B", `Smooth muscle`="#800000",
  `Submucosal Gland`="#F032E6")
for (ds in c("Heart_Datar2026", "Lung_Sikkema2023")) {
  ts <- unique(cells[dataset == ds, mapped_cell_type]); pt <- pretty_cell_type(ts)
  stopifnot(all(pt %in% names(DISTINCT_COLORS)), !anyDuplicated(DISTINCT_COLORS[pt]))
  CELL_COLORS[ts] <- DISTINCT_COLORS[pt]
}
PAGE_W <- 180
ANN <- 32
MOD_W <- 18
MOD_H <- 22
HEAD_H <- 6.5
GAP <- 3
KEY_ROW <- 2.3
mod_theme <- theme_classic(base_family=FONT_FAMILY, base_size=UMAP_FONT) +
  theme(legend.position="none", axis.text=element_blank(), axis.ticks=element_blank(),
        axis.title=element_blank(), axis.line=element_line(linewidth=.2),
        plot.title=element_text(size=UMAP_FONT, face="bold", hjust=.5, lineheight=.9,
                                margin=margin(0,0,.3,0,"mm")),
        plot.margin=margin(.3,.6,.3,.6,"mm"))
brk3 <- function(l) { b <- pretty(l, n=3); b[b >= l[1] & b <= l[2]] }
annotation_map <- function(d, lim) {
  ggplot(d, aes(umap1, umap2, colour=mapped_cell_type)) +
    ggrastr::rasterise(geom_point(size=.12, alpha=.8), dpi=400) +
    scale_colour_manual(values=CELL_COLORS) +
    scale_x_continuous(breaks=brk3(lim$x)) + scale_y_continuous(breaks=brk3(lim$y)) +
    coord_equal(xlim=lim$x, ylim=lim$y, expand=FALSE) +
    labs(title="Cell-type annotation", x="UMAP 1", y="UMAP 2") + map_axes_theme
}
module_map <- function(d, m, i, lim) {
  ggplot() + hex_mean_layers(d$umap1, d$umap2, d[[paste0("activity_", i)]], lim, CELL_COLORS[[m$cell_type]]) +
    coord_equal(xlim=lim$x, ylim=lim$y, expand=FALSE) +
    labs(title=paste0(pretty_cell_type(m$cell_type), "\n", m$LV)) + mod_theme
}
text_mm <- function(x) {
  grDevices::pdf(NULL); on.exit(grDevices::dev.off())
  grid::pushViewport(grid::viewport(gp=grid::gpar(fontfamily=FONT_FAMILY, fontsize=UMAP_FONT)))
  vapply(x, function(s) grid::convertWidth(grid::stringWidth(s), "mm", valueOnly=TRUE), numeric(1))
}
key_layout <- function(types, ncol) {
  per_col <- ceiling(length(types)/ncol)
  k <- data.table(type=types, label=pretty_cell_type(types))[, `:=`(col=(seq_len(.N)-1L) %/% per_col,
                                                                    row=(seq_len(.N)-1L) %% per_col)]
  k[, tw := text_mm(label)]
  cw <- k[, .(mm=max(tw) + 1.3 + 2.2), by=col][order(col)][, start := cumsum(mm) - mm]
  list(k=merge(k, cw[, .(col, start)], by="col"), w=sum(cw$mm) - 2.2 + .6, h=per_col * KEY_ROW)
}
key_plot <- function(types, ncol, w, h) {
  kl <- key_layout(types, ncol); k <- kl$k
  stopifnot(kl$w <= w + .01)
  ggplot(k, aes(start, -row * KEY_ROW)) +
    geom_point(aes(colour=type), size=1.1) +
    geom_text(aes(x=start + 1.3, label=label), hjust=0, size=UMAP_FONT/.pt, family=FONT_FAMILY) +
    scale_colour_manual(values=CELL_COLORS, guide="none") +
    coord_cartesian(xlim=c(-.6, w - .6), ylim=c(-h + 1.15, 1.15), expand=FALSE, clip="off") + theme_void()
}
dataset_block <- function(ds, ncol, nrow, key, key_cols) {
  d <- cells[dataset==ds]; meta <- lvs[dataset==ds][order(plot_order)]; lim <- square_limits(d, pad=.02)
  stopifnot(nrow(meta) <= ncol * nrow)
  st <- lvs[dataset==ds, .(auroc=median(auroc), r=median(assignment_r))]
  grid_w <- ncol * MOD_W; w <- ANN + GAP + grid_w
  types <- sort(unique(d$mapped_cell_type)); kl <- key_layout(types, key_cols); kh <- kl$h; kw <- kl$w
  content_h <- switch(key, under=max(nrow * MOD_H, ANN + 2.5 + kh),
                      below=max(ANN + 1.5, nrow * MOD_H + kh + 1.5), max(ANN + 1.5, nrow * MOD_H))
  h <- HEAD_H + content_h
  ann_stack <- ANN + 1.5 + if (key == "under") 1 + kh else 0
  ann_y <- HEAD_H + (content_h - ann_stack)/2
  items <- list()
  add <- function(p, x, y, pw, ph) items[[length(items) + 1L]] <<- list(p=p, x=x, y=y, w=pw, h=ph)
  add(ggdraw() +
        draw_label(DATASET_LABELS[[ds]], x=.5, y=.75, size=UMAP_FONT, fontface="bold", fontfamily=FONT_FAMILY) +
        draw_label(sprintf("AUROC = %.2f     r = %.2f", st$auroc, st$r), x=.5, y=.25, size=UMAP_FONT,
                   fontfamily=FONT_FAMILY),
      ANN + GAP, 0, grid_w, HEAD_H)
  add(annotation_map(d, lim), 0, ann_y, ANN, ANN + 1.5)
  for (i in seq_len(nrow(meta))) {
    r <- (i - 1L) %/% ncol; cc <- (i - 1L) %% ncol
    add(module_map(d, meta[i], i, lim), ANN + GAP + cc * MOD_W, HEAD_H + r * MOD_H, MOD_W, MOD_H)
  }
  if (key == "below") {
    add(key_plot(types, key_cols, kw, kh), ANN + GAP + (grid_w - kw)/2, HEAD_H + nrow * MOD_H + 1, kw, kh)
  } else if (key == "grid") {
    spare <- ncol * nrow - nrow(meta); first <- nrow(meta)
    stopifnot(spare >= 1L, (first %% ncol) + spare <= ncol, kw <= spare * MOD_W)
    add(key_plot(types, key_cols, kw, kh), ANN + GAP + (first %% ncol) * MOD_W + (spare * MOD_W - kw)/2,
        HEAD_H + (first %/% ncol) * MOD_H + (MOD_H - kh)/2, kw, kh)
  } else {
    stopifnot(kw <= ANN)
    add(key_plot(types, key_cols, kw, kh), (ANN - kw)/2, ann_y + ANN + 2.5, kw, kh)
  }
  list(items=items, w=w, h=h)
}
page_items <- list()
place <- function(block, x0, y0, tag) {
  for (it in block$items) { it$x <- it$x + x0; it$y <- it$y + y0; page_items[[length(page_items) + 1L]] <<- it }
  page_items[[length(page_items) + 1L]] <<- list(
    p=ggdraw() + draw_label(tag, x=0, y=1, hjust=0, vjust=1, size=FS_TAG, fontface="bold", fontfamily=FONT_FAMILY),
    x=max(0, x0 - 1), y=y0, w=5, h=4)
}
panel_A2 <- panel_A + labs(y="Top-1% annotated-cell purity (%)") +
  theme(axis.title.y=element_text(size=UMAP_FONT), axis.text=element_text(size=UMAP_FONT),
        axis.text.x=element_text(angle=40, hjust=1, size=UMAP_FONT))
b_1k1k   <- dataset_block("PBMC_1k1k", 5, 2, "under", 2)
b_mathys <- dataset_block("Brain_Mathys2023", 3, 2, "under", 4)
b_xiong  <- dataset_block("Brain_Xiong2023", 3, 2, "under", 4)
b_heart  <- dataset_block("Heart_Datar2026", 8, 2, "grid", 2)
b_lung   <- dataset_block("Lung_Sikkema2023", 8, 1, "below", 6)
y <- 1
a_w <- PAGE_W - b_1k1k$w - 2 * GAP
page_items[[length(page_items) + 1L]] <- list(p=panel_A2, x=3, y=y + 3, w=a_w - 2, h=b_1k1k$h - 3)
place(list(items=list()), 1, y, "a")
place(b_1k1k, PAGE_W - b_1k1k$w, y, "b"); y <- y + b_1k1k$h + GAP
place(b_mathys, 1, y, "c"); place(b_xiong, PAGE_W - b_xiong$w, y, "d"); y <- y + max(b_mathys$h, b_xiong$h) + GAP
place(b_heart, 1, y, "e"); y <- y + b_heart$h + GAP
place(b_lung, PAGE_W - b_lung$w, y, "f"); y <- y + b_lung$h + 1
PAGE_H <- ceiling(y)
stopifnot(PAGE_H <= NM_HEIGHT, b_mathys$w + b_xiong$w + 1 <= PAGE_W,
          all(sapply(page_items, function(it) it$x >= 0 && it$x + it$w <= PAGE_W + .01)))
page <- ggdraw()
for (it in page_items) page <- page + draw_plot(it$p, x=it$x/PAGE_W, y=1 - (it$y + it$h)/PAGE_H,
                                                 width=it$w/PAGE_W, height=it$h/PAGE_H)
nm_export(list(page),2,"Donor-bulk recovery and cell-type gene-module maps",height=PAGE_H)